In [21]:
import requests
import hashlib
import json
from datetime import datetime, timedelta


def get_md5_encoded_string(phone_number, date, word):
    # Concatenate the input values
    input_string = f"{phone_number}{date}{word}"

    # Encode the string to bytes
    input_bytes = input_string.encode("utf-8")

    # Generate the MD5 hash
    md5_hash = hashlib.md5(input_bytes)

    # Convert the hash to its hexadecimal representation
    md5_hex = md5_hash.hexdigest()

    return md5_hex


# Example usage
phone_number = "18618107293"
date = datetime.now().strftime("%Y%m%d")
word = "login"
md5_encoded_string = get_md5_encoded_string(phone_number, date, word)
print(md5_encoded_string)


def get_token_for_env(env="https://qah5.summerfarm.net"):
    url = f"{env}/openid?phone={phone_number}&sign={md5_encoded_string}"
    print(url)
    token = requests.get(url=url, timeout=12000)
    print(f"token.status_code:{token.status_code}, text:{token.text}")
    try:
        token = json.loads(token.text)
        # print(f"{token}")
        return {"env": f"{env}", "token": token["data"]["token"]}
    except Exception as e:
        print(f"获取Token失败:{url}, {token}, {e}")
        raise e


token_dict = get_token_for_env()

headers = {
    "accept": "application/json, text/plain, */*",
    "token": f"{token_dict['token']}",
    "xm-ab-exp": '[{"experimentId":"product_search_rerank","experimentPlace":"place-of:product_search_rerank","variantId":"V3"},{"experimentId":"new-home","experimentPlace":"place-of:new-home","variantId":"V3"}]',
    "xm-biz": "xm-mall",
    "xm-phone": "18618100000",
    "xm-platform": "web",
    "xm-rqid": "1736314293429-9554778",
    "xm-uid": "350905",
}

print(headers)

526af0f2a2ff58fd79785672056bfae1
https://qah5.summerfarm.net/openid?phone=18618107293&sign=526af0f2a2ff58fd79785672056bfae1
token.status_code:200, text:{"code":"SUCCESS","data":{"address":"浙江杭州市西湖区春申街 西溪花园·凌波苑图012","areaName":"杭州","areaNo":1001,"cbdFlag":false,"changePop":1,"cmbTransferWechatDirectPay":0,"createTime":1716533351000,"deliveryFrequentNew":["2025-01-18"],"displayButton":1,"firstLoginPop":1,"grade":0,"grayscale":1,"groupHeadFlag":0,"groupPurchase":1,"islock":0,"largeAreaNo":1,"mId":349548,"mcontact":"图谱唐","memberList":[{"grade":3,"outTimes":3,"refundAmount":50,"threshold":8000},{"grade":2,"outTimes":2,"refundAmount":20,"threshold":2000},{"grade":1,"outTimes":5,"refundAmount":100,"threshold":150},{"grade":0,"outTimes":0,"refundAmount":0,"threshold":0}],"mergePop":1,"mname":"图谱唐","nextDeliveryDate":"2025-01-18","operateStatus":0,"payChannel":2,"phone":"18618107293","popMerchant":false,"preApprovedMerchant":false,"size":"单店","status":true,"storeCloseOrderTime":"2025-01-16 10:0

In [ ]:
from sls_client import get_sls_raw_data_by_query
from datetime import datetime, timedelta
import pandas as pd

query = """uid:349548 and url:"https://qah5.summerfarm.net" and ap not null"""

from_time = datetime.now() - timedelta(hours=24)
to_time = datetime.now()

all_df = pd.DataFrame()
while True:
    offset = len(all_df)
    print(f"offset:{offset}")
    _df = get_sls_raw_data_by_query(
        from_time=from_time,
        to_time=to_time,
        query=query,
        logstore="fe-test",
        offset=offset,
    )
    if _df.empty:
        break
    all_df = pd.concat([all_df, _df])

In [23]:
import json
from urllib.parse import urlencode

env_list = ["http://localh5", "https://qah5"]


def request_env_data(
    env: str = env_list[1],
    json_data: dict = None,
    params_data: dict = None,
    method: str = "GET",
    api: str = "",
) -> dict:
    if json_data:
        method = "POST"
    host = f"{env}.summerfarm.net{api}"
    if params_data:
        encoded_params = urlencode(params_data)
        host = f"{host}?{encoded_params}"
    result = requests.request(
        method=method, url=host, json=json_data, headers=headers
    ).json()
    if "serverTime" in result:
        del result["serverTime"]
    return result


api_compare_result = []

for index, row in all_df.head(20)[["ai", "uid", "ap"]].iterrows():
    row_dict = json.loads(row.to_dict()["ai"])
    # print(row_dict)
    method = row_dict.get("method").upper()
    request_uri = row["ap"]
    print(request_uri, method)
    json_data = json.loads(row_dict.get("data")) if row_dict.get("data") else None
    params_data = row_dict.get("params")
    print(json_data, params_data)

    new_record = {
        "method": method,
        "request_uri": request_uri,
        "json_data": json_data,
        "params_data": params_data,
    }

    env_results = []
    for env in env_list:
        ret = request_env_data(
            env=env,
            method=method,
            json_data=json_data,
            params_data=params_data,
            api=request_uri,
        )
        new_record[f"result_of_{env}"] = ret
        env_results.append(ret)
    for result in env_results:
        new_record["is_same_result"] = all(
            result == env_results[0] for result in env_results
        )
    api_compare_result.append(new_record)

api_compare_result_df=pd.DataFrame(api_compare_result)
api_compare_result_df

/merchant/job/query/check-job-enabled GET
None None
{'data': True, 'msg': '请求成功', 'status': 200}
/abStrategy/experiments POST
None None
{'data': [{'areaNo': 1001, 'experimentId': 'new-home', 'experimentPlace': 'place-of:new-home', 'mId': 349548, 'variantId': 'V3'}, {'areaNo': 1001, 'experimentId': 'product_search_rerank', 'experimentPlace': 'place-of:product_search_rerank', 'mId': 349548, 'variantId': 'V5'}], 'msg': '请求成功', 'status': 200}
/sub-account/query/timing-old-order-view POST
None None
{'data': [], 'msg': '请求成功', 'status': 200}
/expand_activity/pop_ups GET
None None
{'code': 'SUCCESS', 'msg': '', 'success': True}
/sub-account/query/timing-not-plan-view POST
None None
{'code': 'DEFAULT_FAILED', 'msg': '网络不给力哦', 'success': False}
/order/query/default-contact POST
None None
{'data': {'address': '春申街 西溪花园·凌波苑', 'addressCompletionFlag': 1, 'area': '西湖区', 'city': '杭州市', 'contact': '图谱唐', 'contactId': 347570, 'cutOffTime': 1736992800000, 'deliveryFrequentNew': ['2025-01-18'], 'distanc